# Create dataset with CEOs of US Fortune 500 companies from 2018 to 2026

## Create dataset skeleton with [company name] [year] [CEO]

In [ ]:
import pandas as pd

rows = []

for year in range(2018, 2027):
    filename = f"data/fortune_500/fortune500_USA_{year}.csv"
    df = pd.read_csv(filename)

    for _, row in df.iterrows():
        rows.append({
            "year": year,
            "company_name": row["company"],
            "rank": row["rank"],
            "CEO": ""
        })

skeleton = pd.DataFrame(rows)
skeleton.to_csv("data/ceos.csv", index=False)

print(f"Saved {len(skeleton)} rows to data/ceos.csv")

Saved 4500 rows to data/ceos.csv


## Integrate data from other existing datasets

### 2023

In [ ]:
import pandas as pd
import json

# load the ceos.csv dataset
ceos = pd.read_csv("data/ceos.csv")
ceos["ceo"] = ceos["ceo"].astype(object)

# add the CEO_alt_name column if it doesn't exist yet
if "CEO_alt_name" not in ceos.columns:
    ceos["CEO_alt_name"] = ""
ceos["CEO_alt_name"] = ceos["CEO_alt_name"].astype(object)

# load the json file
with open("data/partial_ceos_data/ceos_usa_2023.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# build a dict: lowercase company name -> (ceo, ceo_alt)
ceo_lookup = {}
for entry in data:
    company = entry["Company"]
    ceo_lookup[company.lower()] = (entry["CEO"], entry["CEO_alt"])

# fill in ceos.csv for year 2023
mask_2023 = ceos["year"] == 2023

for idx in ceos[mask_2023].index:
    company = ceos.loc[idx, "company_name"]
    key = company.lower()
    if key in ceo_lookup:
        ceo_name, ceo_alt = ceo_lookup[key]
        ceos.loc[idx, "ceo"] = ceo_name
        ceos.loc[idx, "CEO_alt_name"] = ceo_alt
    else:
        rank = ceos.loc[idx, "rank"]
        print(f"company {company} (rank {rank}) couldn't be found in the file")

ceos.to_csv("data/ceos.csv", index=False)
print("done")

## Scrape remaining missing data from Wikipedia

### 2026

#### Get Wikipedia URLs and CEOs

In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def get_wikipedia_url(company_name):
    """Convert company name to Wikipedia URL format"""
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"


def extract_ceo_from_infobox(soup):
    """Extract CEO name(s) from Wikipedia infobox"""
    ceos = []
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    rows = infobox.find_all('tr')
    for row in rows:
        header = row.find('th')
        if header:
            header_text = header.get_text().strip().lower()
            if 'key people' in header_text or 'key person' in header_text:
                data = row.find('td')
                if data:
                    text = data.get_text()
                    lines = text.split('\n')
                    for line in lines:
                        if re.search(r'\b(CEO|Chief Executive Officer)\b', line, re.IGNORECASE):
                            name_match = re.match(r'^([^(\[]+)', line)
                            if name_match:
                                name = name_match.group(1).strip()
                                name = re.sub(r'\[.*?\]', '', name)
                                name = name.strip(' ,')
                                if name and len(name) > 2:
                                    ceos.append(name)
    return ', '.join(ceos) if ceos else None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

if "wikipedia_page" not in ceos.columns:
    ceos["wikipedia_page"] = ""
ceos["wikipedia_page"] = ceos["wikipedia_page"].astype(object)

mask_2026 = ceos["year"] == 2026
indices = ceos[mask_2026].index
total = len(indices)

for i, idx in enumerate(indices):
    company = ceos.loc[idx, "company"]
    company_alt = ceos.loc[idx, "company_alt_name"]
    print(f"\n[{i+1}/{total}] Processing: {company}")

    # build list of names to try: company name, then alt name if available
    names_to_try = [company]
    if pd.notna(company_alt) and str(company_alt).strip():
        names_to_try.append(str(company_alt).strip())

    ceo = None
    found_url = None
    for name in names_to_try:
        wiki_url = get_wikipedia_url(name)

        try:
            response = requests.get(wiki_url, headers=HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"  ✗ Request failed for '{name}': {str(e)[:80]}")
            continue

        if response.status_code == 404:
            print(f"  ✗ Wikipedia page not found for '{name}': {wiki_url}")
            continue
        elif response.status_code != 200:
            print(f"  ✗ HTTP {response.status_code} - page not accessible for '{name}'")
            continue

        try:
            soup = BeautifulSoup(response.content, 'html.parser')
            ceo = extract_ceo_from_infobox(soup)
        except Exception as e:
            print(f"  ✗ Unexpected error while parsing page for '{name}': {str(e)[:80]}")
            continue

        if ceo:
            found_url = wiki_url
            break
        else:
            print(f"  ✗ CEO not found in infobox for '{name}'")

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        ceos.loc[idx, "wikipedia_page"] = found_url
        print(f"  ✓ Found CEO: {ceo}")

    ceos.to_csv("data/ceos.csv", index=False)
    time.sleep(1)

print("\ndone")


[1/500] Processing: Visa
  ✗ CEO not found in infobox for 'Visa'

[2/500] Processing: Travelers
  ✗ CEO not found in infobox for 'Travelers'

[3/500] Processing: Pfizer
  ✓ Found CEO: Albert Bourla

[4/500] Processing: Amazon
  ✗ CEO not found in infobox for 'Amazon'

[5/500] Processing: IBM
  ✓ Found CEO: Arvind Krishna

[6/500] Processing: Target
  ✗ CEO not found in infobox for 'Target'

[7/500] Processing: Bank of America
  ✓ Found CEO: Brian Moynihan

[8/500] Processing: Comcast
  ✓ Found CEO: Brian L. Roberts, Michael J. Cavanagh

[9/500] Processing: McKesson
  ✓ Found CEO: Brian S. Tyler

[10/500] Processing: Exelon
  ✓ Found CEO: Calvin Butler

[11/500] Processing: Wells Fargo
  ✓ Found CEO: Steven Black

[12/500] Processing: Costco Wholesale
  ✓ Found CEO: Hamilton E. James

[13/500] Processing: Honeywell International
  ✓ Found CEO: Darius Adamczyk

[14/500] Processing: Boeing
  ✓ Found CEO: Kelly Ortberg

[15/500] Processing: UnitedHealth Group
  ✓ Found CEO: Stephen J. Hem

#### Get CEOs of manually inserted URLs

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def extract_ceo_from_infobox(soup):
    """Extract CEO name(s) from Wikipedia infobox"""
    ceos = []
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    rows = infobox.find_all('tr')
    for row in rows:
        header = row.find('th')
        if header:
            header_text = header.get_text().strip().lower()
            if 'key people' in header_text or 'key person' in header_text:
                data = row.find('td')
                if data:
                    text = data.get_text()
                    lines = text.split('\n')
                    for line in lines:
                        if re.search(r'\b(CEO|Chief Executive Officer)\b', line, re.IGNORECASE):
                            name_match = re.match(r'^([^(\[]+)', line)
                            if name_match:
                                name = name_match.group(1).strip()
                                name = re.sub(r'\[.*?\]', '', name)
                                name = name.strip(' ,')
                                if name and len(name) > 2:
                                    ceos.append(name)
    return ', '.join(ceos) if ceos else None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

mask_2026 = ceos["year"] == 2026
indices = ceos[mask_2026].index

for idx in indices:
    company = ceos.loc[idx, "company"]

    # skip if we already have a ceo
    if pd.notna(ceos.loc[idx, "CEO"]) and str(ceos.loc[idx, "CEO"]).strip():
        continue

    wiki_url = ceos.loc[idx, "wikipedia_page"]
    if pd.isna(wiki_url) or not str(wiki_url).strip():
        print(f"✗ {company}: no wikipedia_page link available")
        continue

    try:
        response = requests.get(wiki_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {company}: HTTP {response.status_code} for {wiki_url}")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        ceo = extract_ceo_from_infobox(soup)
    except Exception as e:
        print(f"✗ {company}: error parsing page - {str(e)[:80]}")
        continue

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        print(f"✓ {company}: found CEO {ceo}")
    else:
        print(f"✗ {company}: CEO not found in infobox")

    time.sleep(1)

ceos.to_csv("data/ceos.csv", index=False)
print("\ndone")

✓ Visa: found CEO Ryan McInerney
✓ Travelers: found CEO Alan D. Schnitzer
✓ Amazon: found CEO Jeff Bezos
✓ Target: found CEO Brian Cornell
✗ Delta Air Lines: CEO not found in infobox
✓ Tesla: found CEO Elon Musk
✓ Coca-Cola: found CEO James Quincey
✗ Norfolk Southern: CEO not found in infobox
✓ Nike: found CEO Philip Knight
✗ General Motors: CEO not found in infobox
✓ Chevron: found CEO Mike Wirth
✓ Danaher: found CEO Steven Rales
✓ Alphabet: found CEO Sundar Pichai
✓ Southern: found CEO Chris Womack
✓ Apple: found CEO Arthur Levinson
✓ Walt Disney: found CEO Josh D'Amaro
✓ UPS: found CEO Carol B. Tomé
✓ RTX: found CEO Christopher T. Calio
✓ Progressive: found CEO Lawton W. Fitt
✓ Caterpillar: found CEO Joe Creed
✓ Eli Lilly: found CEO David A. Ricks
✓ Merck: found CEO Robert M. Davis
✓ Nationwide: found CEO Jeffrey Zellers
✓ Galaxy Digital: found CEO Michael Novogratz
✗ United Airlines Holdings: CEO not found in infobox
✗ Oracle: CEO not found in infobox
✓ HP: found CEO Chip Bergh
✓ I

#### Correction of CEOs with more accurate scraping

In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}


def replace_br_with_newline(tag):
    for br in tag.find_all("br"):
        br.replace_with("\n")


def extract_ceo_from_infobox(soup):
    """Extract CEO name from Wikipedia infobox, handling different 'Key people' layouts"""
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    for row in infobox.find_all('tr'):
        header = row.find('th')
        if not header:
            continue
        header_text = header.get_text().strip().lower()
        if 'key people' not in header_text and 'key person' not in header_text:
            continue

        data = row.find('td')
        if not data:
            continue

        # build a list of text segments: one per <li>, or split by <br> if no list
        lis = data.find_all('li')
        if lis:
            segments = [li.get_text(" ", strip=True) for li in lis]
        else:
            replace_br_with_newline(data)
            segments = [s.strip() for s in data.get_text().split('\n') if s.strip()]

        # merge segments that are only a parenthetical title into the previous segment
        merged = []
        for seg in segments:
            if seg.startswith('(') and merged:
                merged[-1] = merged[-1] + ' ' + seg
            else:
                merged.append(seg)

        # find segments mentioning CEO
        ceo_names = []
        for seg in merged:
            if re.search(r'\bCEO\b', seg, re.IGNORECASE) or re.search(r'chief executive officer', seg, re.IGNORECASE):
                name_match = re.match(r'^([^(]+)', seg)
                if name_match:
                    name = re.sub(r'\[.*?\]', '', name_match.group(1)).strip(' ,')
                    if name and len(name) > 2:
                        ceo_names.append(name)

        return ', '.join(ceo_names) if ceo_names else None

    return None


ceos = pd.read_csv("data/ceos.csv")
ceos["CEO"] = ceos["CEO"].astype(object)

mask_2026 = ceos["year"] == 2026
indices = ceos[mask_2026].index

for idx in indices:
    company = ceos.loc[idx, "company"]
    wiki_url = ceos.loc[idx, "wikipedia_page"]

    if pd.isna(wiki_url) or not str(wiki_url).strip():
        continue

    try:
        response = requests.get(wiki_url, headers=HEADERS, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"✗ {company}: request failed - {str(e)[:80]}")
        continue

    if response.status_code != 200:
        print(f"✗ {company}: HTTP {response.status_code} for {wiki_url}")
        continue

    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        ceo = extract_ceo_from_infobox(soup)
    except Exception as e:
        print(f"✗ {company}: error parsing page - {str(e)[:80]}")
        continue

    if ceo:
        ceos.loc[idx, "CEO"] = ceo
        print(f"✓ {company}: found CEO {ceo}")
    else:
        print(f"✗ {company}: CEO not found in infobox")

    time.sleep(1)

ceos.to_csv("data/ceos.csv", index=False)
print("\ndone")

✓ Amazon: found CEO Andy Jassy
✓ Walmart: found CEO John Furner
✓ UnitedHealth Group: found CEO Stephen J. Hemsley, Tim Noel
✓ Apple: found CEO Tim Cook, John Ternus
✓ Alphabet: found CEO Sundar Pichai
✓ CVS Health: found CEO David Joyner
✓ Berkshire Hathaway: found CEO Greg Abel
✓ McKesson: found CEO Brian S. Tyler
✓ ExxonMobil Holdings: found CEO Darren Woods
✓ Cencora: found CEO Robert Mauch
✓ Microsoft: found CEO Satya Nadella
✓ JPMorgan Chase: found CEO Jamie Dimon
✓ Costco Wholesale: found CEO Ron Vachris
✓ Cigna Group: found CEO Brian Evanko
✓ Cardinal Health: found CEO Jason Hollar, Deborah Weitzman, Stephen Mason
✓ Nvidia: found CEO Jensen Huang
✓ Meta Platforms: found CEO Mark Zuckerberg
✓ Elevance Health: found CEO Gail Koziara Boudreaux
✓ Centene: found CEO Sarah London
✓ Bank of America: found CEO Brian Moynihan
✓ Chevron: found CEO Mike Wirth
✓ Ford Motor: found CEO Jim Farley
✓ General Motors: found CEO Alfred P. Sloan CEO
✓ Citigroup: found CEO Jane Fraser
✓ Home Depot: